# Model Context Protocol (MCP)

The task, titled *MCP Ping‑Pong*, asks you to build a minimal Model Context Protocol (MCP) server and a simple client that communicate via the standard input/output (STDIO). The server will expose a single weather tool and a resource that lists supported cities. The client will connect to the server, query the available resources and tools, and fetch the weather for a chosen city.

The following sections present the code for the `server.py` and `client.py` scripts as well as an example of the expected output when running the client.

## server.py

The server is built using the `FastMCP` class from the `mcp` library. We create a server named `WeatherDemo` and register a single tool called `get_weather`. This tool accepts a city name and returns a dictionary with static weather information for supported cities (Paris, London and New York). We also register a resource `cities://list` that returns a newline-separated list of the supported cities. The server listens on standard input/output using the `run_stdio()` method.

In [ ]:
# server.py
from mcp.server.fastmcp import FastMCP

# Create an MCP server with a name
mcp = FastMCP('WeatherDemo')

# Static weather data for a few cities
_WEATHER_DATA = {
    'Paris': {'temperature': 20, 'conditions': 'Sunny'},
    'London': {'temperature': 15, 'conditions': 'Cloudy'},
    'New York': {'temperature': 25, 'conditions': 'Partly cloudy'},
}

@mcp.tool()
def get_weather(city: str) -> dict:
    # Return static weather data for the given city.
    data = _WEATHER_DATA.get(city)
    if data is None:
        raise ValueError(f'Unsupported city: {city}')
    return data

@mcp.resource('cities://list')
def cities_list() -> str:
    # Return a newline-separated list of supported city names.
    return '
'.join(_WEATHER_DATA.keys())

if __name__ == '__main__':
    # Start the server and listen via standard I/O
    mcp.run_stdio()


## client.py

The client connects to the MCP server using the `StdioServerParameters` class from `mcp.client.stdio`. It lists the resources and tools exposed by the server, reads the list of supported cities from `cities://list`, and then calls the `get_weather` tool for one of the cities (Paris).

In [ ]:
# client.py
import asyncio
from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters

async def main():
    # Define how to start the server via the mcp CLI; assumes server.py is in the same directory.
    server_params = StdioServerParameters(command='mcp', args=['run', 'server.py'])

    # Connect to the server via stdio
    async with ClientSession() as session:
        async with stdio_client(server_params=server_params, client_session=session) as conn:
            # List available resources and tools
            resources = await conn.list_resources()
            tools = await conn.list_tools()
            print('Resources:', resources)
            print('Tools:', tools)

            # Read the list of supported cities
            cities = await conn.read_resource('cities://list')
            print('Cities:')
            print(cities)

            # Call the get_weather tool for Paris
            weather = await conn.call_tool('get_weather', {'city': 'Paris'})
            print('Weather for Paris:')
            print(weather)

# Run the asynchronous main function
if __name__ == '__main__':
    asyncio.run(main())


### Expected Output

When running the client script (after ensuring that `mcp` is installed and the server code is available), you should see output similar to the following:

``
Resources: ['cities://list']
Tools: ['get_weather']
Cities:
Paris
London
New York
Weather for Paris:
{'temperature': 20, 'conditions': 'Sunny'}
``
